
## 01_simulate_landing

Simulates real-time event data arriving in Bronze.

In production:
  - Files arrive from Kafka → S3/ADLS automatically
  - Cloud storage sends event notification to Auto Loader
  - Auto Loader picks up file immediately

Here we simulate by:
  - Downloading GitHub Archive hourly JSON files
  - Dropping them into Bronze one at a time
  - Pausing between drops (mimics time passing)

GitHub Archive format:
  - NDJSON (Newline Delimited JSON) compressed with gzip
  - One JSON object per line = one GitHub event
  - Event types: PushEvent, PullRequestEvent, WatchEvent, IssueEvent
  - Each file = ~1 hour of ALL public GitHub activity

In [0]:
%python

# Define config variables inline
BASE_PATH = "/Volumes/workspace/default/github_streaming"
BRONZE_PATH = f"{BASE_PATH}/bronze/events"
GITHUB_ARCHIVE_BASE = "https://data.gharchive.org"
LANDING_HOURS = ["0", "1", "2", "3"]
PIPELINE = {
    "source_date": "2024-01-01",
    "write_mode": "append",
}

import requests
import gzip
import shutil
import os
import tempfile
import time
from datetime import datetime

print("Libraries loaded ✓")
print(f"Landing date : {PIPELINE['source_date']}")
print(f"Hours to land: {LANDING_HOURS}")

In [0]:
%python

# ── Download and land one hourly file ────────────────────────
# STUDY NOTE:
#   GitHub Archive files are gzip compressed NDJSON.
#   We decompress them before landing in Bronze because:
#   1. Spark reads plain JSON/NDJSON natively
#   2. Auto Loader handles JSON schema inference on plain files
#   3. Compressed files need extra config in Auto Loader
#
#   NDJSON vs JSON:
#   Regular JSON → one big object/array per file
#   NDJSON       → one JSON object per LINE (easier to stream)
#   {"id": 1, "type": "PushEvent", ...}
#   {"id": 2, "type": "WatchEvent", ...}
#   Each line is independent — perfect for event streaming

def download_and_land(date: str, hour: str) -> str:
    """
    Downloads one hourly GitHub Archive file and lands
    it as decompressed NDJSON in the Bronze path.
    Returns the Bronze file path.
    """
    url = f"{GITHUB_ARCHIVE_BASE}/{date}-{hour}.json.gz"
    print(f"\nDownloading: {url}")

    # Download compressed file to temp location
    tmp_dir  = tempfile.mkdtemp()
    gz_path  = os.path.join(tmp_dir, f"{date}-{hour}.json.gz")
    json_path = os.path.join(tmp_dir, f"{date}-{hour}.json")

    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(gz_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)

    gz_size = os.path.getsize(gz_path) / (1024 * 1024)
    print(f"  Downloaded : {gz_size:.1f} MB (compressed)")

    # Decompress gzip → plain NDJSON
    with gzip.open(gz_path, "rb") as f_in:
        with open(json_path, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)

    json_size = os.path.getsize(json_path) / (1024 * 1024)
    print(f"  Decompressed: {json_size:.1f} MB (plain JSON)")

    # Copy to Bronze path
    bronze_file = f"{BRONZE_PATH}/{date}-{hour}.json"
    shutil.copy(json_path, bronze_file)

    # Cleanup temp
    shutil.rmtree(tmp_dir)

    print(f"  ✓ Landed in Bronze: {bronze_file}")
    return bronze_file

In [0]:
%python

# ── Land files simulating time-based arrival ──────────────────
# STUDY NOTE:
#   We land files one at a time with a small pause.
#   This is crucial for Auto Loader demonstration:
#   - Land hour 0 → Auto Loader detects it → processes
#   - Land hour 1 → Auto Loader detects NEW file → processes
#   - Land hour 2 → same pattern
#
#   In production the pause is removed — files arrive
#   continuously and Auto Loader keeps up in real time.
#
#   trigger(availableNow=True) in Auto Loader means:
#   "process all files currently in Bronze, then stop"
#   This is called "incremental batch" mode —
#   best of both worlds: streaming detection + batch processing

landed_files = []
date = PIPELINE["source_date"]

print(f"Starting simulated landing for {date}")
print(f"Landing {len(LANDING_HOURS)} hourly files...\n")

for hour in LANDING_HOURS:
    try:
        path = download_and_land(date, hour)
        landed_files.append(path)

        # Count events in this file
        df_check = spark.read.json(path)
        event_count = df_check.count()
        print(f"  Events in hour {hour}: {event_count:,}")

    except Exception as e:
        print(f"  ✗ Hour {hour} failed: {e}")

print(f"\n{'='*50}")
print(f"Landing complete")
print(f"Files landed : {len(landed_files)}")

In [0]:
%python

# ── Verify what landed in Bronze ─────────────────────────────
# STUDY NOTE:
#   This is the state Auto Loader will "see" when it runs.
#   Every file here is a candidate for processing.
#   Auto Loader compares this list to its checkpoint
#   to determine which files are NEW.

print("Bronze directory contents:")
print("-" * 50)

bronze_files = dbutils.fs.ls(BRONZE_PATH)
total_size = 0

for f in bronze_files:
    size_mb = f.size / (1024 * 1024)
    total_size += size_mb
    print(f"  {f.name:<30} {size_mb:.1f} MB")

print("-" * 50)
print(f"Total files : {len(bronze_files)}")
print(f"Total size  : {total_size:.1f} MB")
print(f"\nReady for Auto Loader in 02_autoloader_silver ✓")

In [0]:
%python

# ── Preview raw GitHub event structure ───────────────────────
# STUDY NOTE:
#   GitHub events are NESTED JSON — very different from
#   the flat Parquet we used in Project 1.
#
#   Nested JSON means values can themselves be objects:
#   {
#     "id": "123",
#     "type": "PushEvent",
#     "actor": {                  ← nested object
#       "id": 456,
#       "login": "user123"        ← field INSIDE nested object
#     },
#     "repo": {                   ← another nested object
#       "id": 789,
#       "name": "user123/myrepo"
#     },
#     "payload": {                ← deeply nested
#       "commits": [...]          ← array inside object
#     }
#   }
#
#   In Silver we'll FLATTEN this structure —
#   extract key fields to top level for easier querying.
#   This is called "schema flattening" or "denormalization".

# Read first landed file
first_file = f"{BRONZE_PATH}/{PIPELINE['source_date']}-{LANDING_HOURS[0]}.json"
df_raw = spark.read.json(first_file)

print(f"Raw schema from Bronze (nested JSON):")
df_raw.printSchema()

print(f"\nTotal fields at top level: {len(df_raw.columns)}")
print(f"Sample event types:")
display(
    df_raw.groupBy("type")
          .count()
          .orderBy("count", ascending=False)
          .limit(10)
)

In [0]:
%python

# Define config variables inline
BASE_PATH = "/Volumes/workspace/default/github_streaming"
BRONZE_PATH = f"{BASE_PATH}/bronze/events"
GITHUB_ARCHIVE_BASE = "https://data.gharchive.org"
PIPELINE = {
    "source_date": "2024-01-01",
    "write_mode": "append",
}

import requests, gzip, shutil, os, tempfile

# Land ONE new hour — hour 4
date = PIPELINE["source_date"]
hour = "4"

url = f"{GITHUB_ARCHIVE_BASE}/{date}-{hour}.json.gz"
print(f"Downloading hour {hour}...")

tmp_dir   = tempfile.mkdtemp()
gz_path   = os.path.join(tmp_dir, f"{date}-{hour}.json.gz")
json_path = os.path.join(tmp_dir, f"{date}-{hour}.json")

with requests.get(url, stream=True) as r:
    r.raise_for_status()
    with open(gz_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

with gzip.open(gz_path, "rb") as f_in:
    with open(json_path, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)

bronze_file = f"{BRONZE_PATH}/{date}-{hour}.json"
shutil.copy(json_path, bronze_file)
shutil.rmtree(tmp_dir)

print(f"✓ Hour {hour} landed: {bronze_file}")
print(f"\nBronze now has:")
for f in dbutils.fs.ls(BRONZE_PATH):
    print(f"  {f.name}")